# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_07 — Purged K-Fold and Embargo Cross-Validation

### Research question

How do we evaluate time-series trading models when labels overlap through time, so that ordinary random K-fold cross-validation leaks future information?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
05_05_bayesian_strategy_calibration
05_06_performance_haircut_and_deflated_sharpe
```

This notebook focuses on one of the most common hidden sources of overfitting in financial machine learning:

> overlapping labels.

If a label at time $t$ uses returns from $t$ to $t+H$, then nearby labels share future information. Random K-fold cross-validation mixes these overlapping labels between train and test.

That is leakage.

This notebook covers:

1. event labels and label horizons;
2. overlapping forward-return labels;
3. why random K-fold leaks;
4. why ordinary time-series split may still leak;
5. event intervals $[t_0,t_1]$;
6. purging overlapping training observations;
7. embargo after validation folds;
8. purged K-fold implementation;
9. feature availability checks;
10. random K-fold versus chronological split versus purged K-fold;
11. intentionally leaky feature demonstration;
12. model training inside each fold;
13. cross-validation score comparison;
14. fold-level diagnostics;
15. sample concurrency;
16. sample uniqueness;
17. embargo sensitivity;
18. strategy backtest from cross-validated predictions;
19. governance flags;
20. saved outputs and manifest.

The key idea:

> If labels overlap in time, train/test rows can be different rows and still share the same future return information.

## 1. Overlapping labels

Suppose the label is a 10-day forward return:

$$
y_t = \sum_{i=1}^{10} r_{t+i}
$$

Then $y_t$ and $y_{t+1}$ share 9 out of 10 future returns.

If $y_t$ is in the training set and $y_{t+1}$ is in the validation set, the model has indirectly seen much of the validation outcome.

That is why ordinary K-fold is dangerous for financial labels.

## 2. Event intervals

Each labelled observation is an event:

$$
[t_0, t_1]
$$

where:

- $t_0$: prediction time;
- $t_1$: label end time.

For an $H$-day forward-return label:

$$
t_1 = t_0 + H
$$

A training event overlaps a validation event if:

$$
t_{0,train} \le t_{1,val} \quad\text{and}\quad t_{1,train} \ge t_{0,val}
$$

Purging removes overlapping training events.

## 3. Embargo

Even after purging, adjacent observations immediately after a validation fold can be dangerous because features, labels, or market microstructure effects can bleed across the boundary.

Embargo removes training samples for a short period after the validation fold:

$$
[t_{val,end}, t_{val,end} + embargo]
$$

Purging handles direct label overlap.

Embargo handles boundary contamination.

## 4. Cross-validation methods compared

We compare:

### Random K-fold

Fast, familiar, and usually wrong for time-series labels.

### Chronological K-fold

Uses contiguous test blocks, but does not necessarily purge overlapping labels.

### Purged K-fold

Uses contiguous test blocks and removes overlapping training events.

### Purged K-fold with embargo

Adds a buffer after each test block.

This is the main target method.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class PurgedKFoldConfig:
    n_days: int = 2_400
    annualisation: int = 252
    seed: int = 42
    label_horizon: int = 10
    n_splits: int = 5
    embargo_pct: float = 0.02
    min_train_size: int = 200
    transaction_cost_bps: float = 4.0
    target_gross: float = 1.0
    cvar_alpha: float = 0.95

config = PurgedKFoldConfig()
config

## 6. Simulate market data with weak predictability

We simulate a single tradable asset plus market features.

The asset has:

- volatility regimes;
- stress events;
- weak momentum/reversal structure;
- noisy returns.

This creates a realistic classification problem: predict whether the next $H$-day return is positive.

In [ ]:
def simulate_time_series_data(config: PurgedKFoldConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2015-01-01", periods=config.n_days)

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    returns = np.zeros(config.n_days)
    market = np.zeros(config.n_days)
    rates = np.zeros(config.n_days)
    commodity = np.zeros(config.n_days)
    vol_state = np.zeros(config.n_days)

    for t in range(config.n_days):
        if t > 0:
            if regime[t - 1] == 0:
                regime[t] = rng.choice([0, 1], p=[0.985, 0.015])
            else:
                regime[t] = rng.choice([0, 1], p=[0.060, 0.940])

        vol_mult = 1.0 if regime[t] == 0 else 2.5

        market_shock = rng.standard_t(df=6) * 0.009 * vol_mult
        rates_shock = rng.standard_t(df=6) * 0.0035 * vol_mult
        commodity_shock = rng.standard_t(df=6) * 0.010 * vol_mult

        u = rng.uniform()
        stress = 0.0
        if u < 0.008:
            stress_type[t] = "equity_crash"
            stress = -rng.uniform(0.025, 0.080)
            market_shock += stress
            rates_shock += rng.uniform(0.002, 0.010)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            commodity_shock += rng.uniform(0.025, 0.080)
            rates_shock -= rng.uniform(0.004, 0.018)
            market_shock -= rng.uniform(0.005, 0.030)

        market[t] = market_shock
        rates[t] = rates_shock
        commodity[t] = commodity_shock
        vol_state[t] = vol_mult

        # Weak true structure.
        momentum_effect = 0.0
        reversal_effect = 0.0
        stress_effect = 0.0

        if t >= 20:
            momentum_effect = 0.06 * np.mean(returns[t-20:t])
        if t >= 5:
            reversal_effect = -0.04 * np.mean(returns[t-5:t])
        if t > 0 and regime[t - 1] == 1:
            stress_effect = 0.0004

        returns[t] = (
            0.00012
            + 0.45 * market_shock
            - 0.25 * rates_shock
            + 0.20 * commodity_shock
            + momentum_effect
            + reversal_effect
            + stress_effect
            + rng.standard_t(df=5) * 0.006 * vol_mult
        )

    df = pd.DataFrame({
        "asset_return": returns,
        "market_return": market,
        "rates_return": rates,
        "commodity_return": commodity,
        "regime": regime,
        "stress_type": stress_type,
        "vol_state": vol_state,
    }, index=dates)

    df["price"] = 100.0 * np.exp(np.log1p(df["asset_return"]).cumsum())
    return df

data = simulate_time_series_data(config)

data.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index, data["price"])
plt.title("Synthetic Asset Price")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(data.index, data["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 7. Feature engineering with availability discipline

All features at $t$ must be known at $t$.

We use only lagged or contemporaneously available data:

- lagged returns;
- rolling momentum;
- rolling volatility;
- rolling skew;
- market/rates/commodity features;
- regime proxy.

We also create one intentionally leaky feature for demonstration.

In [ ]:
def build_features(data: pd.DataFrame, config: PurgedKFoldConfig):
    df = data.copy()

    features = pd.DataFrame(index=df.index)

    for lag in [1, 2, 3, 5, 10]:
        features[f"asset_return_lag_{lag}"] = df["asset_return"].shift(lag)
        features[f"market_return_lag_{lag}"] = df["market_return"].shift(lag)

    for window in [5, 10, 20, 63]:
        features[f"momentum_{window}"] = df["asset_return"].rolling(window).sum().shift(1)
        features[f"volatility_{window}"] = df["asset_return"].rolling(window).std().shift(1)
        features[f"market_momentum_{window}"] = df["market_return"].rolling(window).sum().shift(1)
        features[f"commodity_momentum_{window}"] = df["commodity_return"].rolling(window).sum().shift(1)

    features["rates_return_lag_1"] = df["rates_return"].shift(1)
    features["commodity_return_lag_1"] = df["commodity_return"].shift(1)
    features["regime_lag_1"] = df["regime"].shift(1)
    features["vol_state_lag_1"] = df["vol_state"].shift(1)

    # Intentionally leaky feature: future return over the label horizon.
    features["LEAKY_future_return_horizon"] = df["asset_return"].shift(-1).rolling(config.label_horizon).sum().shift(-(config.label_horizon - 1))

    return features

features_all = build_features(data, config)

features_all.head()

## 8. Build overlapping event labels

Label:

$$
y_t = 1 \quad\text{if}\quad \sum_{i=1}^{H} r_{t+i} > 0
$$

Prediction time:

$$
t_0=t
$$

Label end time:

$$
t_1=t+H
$$

The event table stores $t_0$, $t_1$, forward return, and binary target.

In [ ]:
def build_event_labels(data: pd.DataFrame, features: pd.DataFrame, config: PurgedKFoldConfig, include_leaky=False):
    forward_return = data["asset_return"].shift(-1).rolling(config.label_horizon).sum().shift(-(config.label_horizon - 1))
    label = (forward_return > 0).astype(float)

    event_end = pd.Series(index=data.index, dtype="datetime64[ns]")
    for i, timestamp in enumerate(data.index):
        end_i = i + config.label_horizon
        if end_i < len(data.index):
            event_end.loc[timestamp] = data.index[end_i]

    feature_cols = list(features.columns)
    if not include_leaky:
        feature_cols = [c for c in feature_cols if not c.startswith("LEAKY")]

    dataset = features[feature_cols].copy()
    dataset["forward_return"] = forward_return
    dataset["label"] = label
    dataset["t0"] = dataset.index
    dataset["t1"] = event_end

    dataset = dataset.dropna().copy()
    dataset["label"] = dataset["label"].astype(int)

    return dataset, feature_cols

dataset_clean, feature_cols_clean = build_event_labels(data, features_all, config, include_leaky=False)
dataset_leaky, feature_cols_leaky = build_event_labels(data, features_all, config, include_leaky=True)

dataset_clean.head(), len(dataset_clean), len(feature_cols_clean), len(feature_cols_leaky)

## 9. Visualise label overlap

Because labels span $H$ days, many events are active at the same time.

Concurrency counts how many event intervals cover each date.

High concurrency means high overlap.

In [ ]:
def compute_concurrency(event_table: pd.DataFrame, index: pd.Index):
    concurrency = pd.Series(0, index=index, dtype=float)

    for _, row in event_table.iterrows():
        t0 = row["t0"]
        t1 = row["t1"]
        if pd.isna(t1):
            continue
        concurrency.loc[(concurrency.index >= t0) & (concurrency.index <= t1)] += 1

    return concurrency

concurrency = compute_concurrency(dataset_clean, data.index)

concurrency_report = pd.Series({
    "mean_concurrency": concurrency.mean(),
    "median_concurrency": concurrency.median(),
    "max_concurrency": concurrency.max(),
    "label_horizon": config.label_horizon,
})

concurrency_report

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(concurrency.index, concurrency)
plt.title("Event Concurrency from Overlapping Labels")
plt.xlabel("Date")
plt.ylabel("Number of active labels")
plt.show()

## 10. Purging and embargo functions

Purging removes training events whose intervals overlap with the validation interval.

Embargo removes training events immediately after the validation interval.

In [ ]:
def get_test_interval(event_table, test_indices):
    test_events = event_table.iloc[test_indices]
    test_start = test_events["t0"].min()
    test_end = test_events["t1"].max()
    return test_start, test_end

def purge_train_indices(event_table, train_indices, test_indices):
    test_start, test_end = get_test_interval(event_table, test_indices)
    train_events = event_table.iloc[train_indices]

    overlaps = (
        (train_events["t0"] <= test_end)
        & (train_events["t1"] >= test_start)
    )

    keep = ~overlaps
    return train_events.index[keep].to_numpy()

def embargo_train_indices(event_table, train_timestamp_index, test_indices, embargo_pct):
    if embargo_pct <= 0:
        return train_timestamp_index

    n = len(event_table)
    embargo_size = int(np.ceil(n * embargo_pct))

    test_end_pos = max(test_indices)
    embargo_start_pos = test_end_pos + 1
    embargo_end_pos = min(n - 1, test_end_pos + embargo_size)

    if embargo_start_pos > embargo_end_pos:
        return train_timestamp_index

    embargo_timestamps = event_table.iloc[embargo_start_pos:embargo_end_pos + 1].index
    return np.array([idx for idx in train_timestamp_index if idx not in set(embargo_timestamps)])

def timestamp_index_to_pos(event_table, timestamps):
    loc = event_table.index.get_indexer(timestamps)
    return loc[loc >= 0]

## 11. Cross-validation splitters

We implement:

1. random K-fold;
2. chronological K-fold;
3. purged K-fold;
4. purged K-fold plus embargo.

In [ ]:
def random_kfold_splits(event_table, n_splits, seed):
    rng = np.random.default_rng(seed)
    n = len(event_table)
    indices = np.arange(n)
    rng.shuffle(indices)

    folds = np.array_split(indices, n_splits)
    splits = []

    for fold_id, test_idx in enumerate(folds):
        train_idx = np.setdiff1d(indices, test_idx)
        splits.append({
            "fold_id": fold_id,
            "train_idx": np.sort(train_idx),
            "test_idx": np.sort(test_idx),
            "method": "random_kfold",
        })

    return splits

def chronological_kfold_splits(event_table, n_splits):
    n = len(event_table)
    indices = np.arange(n)
    folds = np.array_split(indices, n_splits)

    splits = []
    for fold_id, test_idx in enumerate(folds):
        train_idx = np.setdiff1d(indices, test_idx)
        splits.append({
            "fold_id": fold_id,
            "train_idx": np.sort(train_idx),
            "test_idx": np.sort(test_idx),
            "method": "chronological_kfold_no_purge",
        })
    return splits

def purged_kfold_splits(event_table, n_splits, embargo_pct=0.0):
    n = len(event_table)
    indices = np.arange(n)
    folds = np.array_split(indices, n_splits)

    splits = []

    for fold_id, test_idx in enumerate(folds):
        raw_train_idx = np.setdiff1d(indices, test_idx)

        purged_train_timestamps = purge_train_indices(event_table, raw_train_idx, test_idx)
        embargoed_train_timestamps = embargo_train_indices(event_table, purged_train_timestamps, test_idx, embargo_pct)
        train_idx = timestamp_index_to_pos(event_table, embargoed_train_timestamps)

        splits.append({
            "fold_id": fold_id,
            "train_idx": np.sort(train_idx),
            "test_idx": np.sort(test_idx),
            "method": "purged_kfold_embargo" if embargo_pct > 0 else "purged_kfold_no_embargo",
            "embargo_pct": embargo_pct,
        })

    return splits

splits_random = random_kfold_splits(dataset_clean, config.n_splits, config.seed)
splits_chrono = chronological_kfold_splits(dataset_clean, config.n_splits)
splits_purged = purged_kfold_splits(dataset_clean, config.n_splits, embargo_pct=0.0)
splits_purged_embargo = purged_kfold_splits(dataset_clean, config.n_splits, embargo_pct=config.embargo_pct)

pd.DataFrame([
    {"method": s["method"], "fold_id": s["fold_id"], "n_train": len(s["train_idx"]), "n_test": len(s["test_idx"])}
    for s in splits_purged_embargo
])

## 12. Split overlap diagnostics

We verify how much training data is removed by purging and embargo.

We also explicitly count remaining interval overlaps.

In [ ]:
def count_train_test_overlaps(event_table, train_idx, test_idx):
    train_events = event_table.iloc[train_idx]
    test_events = event_table.iloc[test_idx]

    count = 0
    for _, tr in train_events.iterrows():
        overlaps = (
            (test_events["t0"] <= tr["t1"])
            & (test_events["t1"] >= tr["t0"])
        )
        count += int(overlaps.sum())

    return count

def split_diagnostics(event_table, splits, raw_reference_splits=None):
    rows = []

    for s in splits:
        raw_n_train = len(event_table) - len(s["test_idx"])
        overlaps = count_train_test_overlaps(event_table, s["train_idx"], s["test_idx"])

        rows.append({
            "method": s["method"],
            "fold_id": s["fold_id"],
            "n_train": len(s["train_idx"]),
            "n_test": len(s["test_idx"]),
            "raw_n_train": raw_n_train,
            "n_removed": raw_n_train - len(s["train_idx"]),
            "removed_pct": 1 - len(s["train_idx"]) / raw_n_train if raw_n_train > 0 else np.nan,
            "remaining_train_test_interval_overlaps": overlaps,
            "test_start": event_table.iloc[s["test_idx"]]["t0"].min(),
            "test_end": event_table.iloc[s["test_idx"]]["t1"].max(),
        })

    return pd.DataFrame(rows)

split_diag = pd.concat([
    split_diagnostics(dataset_clean, splits_random),
    split_diagnostics(dataset_clean, splits_chrono),
    split_diagnostics(dataset_clean, splits_purged),
    split_diagnostics(dataset_clean, splits_purged_embargo),
], ignore_index=True)

split_diag

## 13. Model training inside each fold

Important rule:

> Preprocessing must be fit only on training data inside each fold.

We use logistic regression when scikit-learn is available.

A small fallback logistic classifier is included otherwise.

In [ ]:
class SimpleLogisticClassifier:
    def __init__(self, lr=0.05, l2=1.0, n_iter=500):
        self.lr = lr
        self.l2 = l2
        self.n_iter = n_iter
        self.mean_ = None
        self.std_ = None
        self.coef_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        Xs = (X - self.mean_) / self.std_

        Xb = np.column_stack([np.ones(len(Xs)), Xs])
        w = np.zeros(Xb.shape[1])

        for _ in range(self.n_iter):
            z = Xb @ w
            p = 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
            grad = Xb.T @ (p - y) / len(y)
            grad[1:] += self.l2 * w[1:] / len(y)
            w -= self.lr * grad

        self.coef_ = w
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        Xs = (X - self.mean_) / self.std_
        Xb = np.column_stack([np.ones(len(Xs)), Xs])
        z = Xb @ self.coef_
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
        return np.column_stack([1 - p, p])

def make_classifier():
    if SKLEARN_AVAILABLE:
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, C=0.5, solver="lbfgs")),
        ])
    return SimpleLogisticClassifier(lr=0.05, l2=1.0, n_iter=800)

def safe_auc(y_true, pred_prob):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    if SKLEARN_AVAILABLE:
        return float(roc_auc_score(y_true, pred_prob))
    # Manual rank AUC fallback.
    order = np.argsort(pred_prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(pred_prob) + 1)
    pos = y_true == 1
    n_pos = pos.sum()
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    auc = (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(auc)

def safe_log_loss(y_true, pred_prob):
    p = np.clip(pred_prob, 1e-6, 1 - 1e-6)
    y = np.asarray(y_true)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

## 14. Cross-validation evaluator

For each fold:

1. fit preprocessing and model on train only;
2. predict probabilities on test;
3. compute accuracy, AUC, log loss;
4. save predictions and fold diagnostics.

In [ ]:
def evaluate_cv(event_table, feature_cols, splits, method_name):
    X = event_table[feature_cols].astype(float)
    y = event_table["label"].astype(int)

    pred_rows = []
    fold_rows = []

    for s in splits:
        fold_id = s["fold_id"]
        train_idx = s["train_idx"]
        test_idx = s["test_idx"]

        if len(train_idx) < config.min_train_size:
            continue

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_test = y.iloc[test_idx]

        model = make_classifier()
        model.fit(X_train, y_train)
        p_test = model.predict_proba(X_test)[:, 1]

        pred_label = (p_test >= 0.5).astype(int)

        fold_auc = safe_auc(y_test, p_test)
        fold_acc = float((pred_label == y_test.to_numpy()).mean())
        fold_loss = safe_log_loss(y_test, p_test)

        fold_rows.append({
            "method": method_name,
            "fold_id": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "auc": fold_auc,
            "accuracy": fold_acc,
            "log_loss": fold_loss,
            "positive_rate_train": float(y_train.mean()),
            "positive_rate_test": float(y_test.mean()),
        })

        fold_pred = pd.DataFrame({
            "timestamp": event_table.iloc[test_idx].index,
            "method": method_name,
            "fold_id": fold_id,
            "y_true": y_test.to_numpy(),
            "pred_prob": p_test,
            "pred_label": pred_label,
            "forward_return": event_table.iloc[test_idx]["forward_return"].to_numpy(),
            "t0": event_table.iloc[test_idx]["t0"].to_numpy(),
            "t1": event_table.iloc[test_idx]["t1"].to_numpy(),
        })
        pred_rows.append(fold_pred)

    preds = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
    folds = pd.DataFrame(fold_rows)

    return preds, folds

pred_random, folds_random = evaluate_cv(dataset_clean, feature_cols_clean, splits_random, "random_kfold")
pred_chrono, folds_chrono = evaluate_cv(dataset_clean, feature_cols_clean, splits_chrono, "chronological_kfold_no_purge")
pred_purged, folds_purged = evaluate_cv(dataset_clean, feature_cols_clean, splits_purged, "purged_kfold_no_embargo")
pred_purged_embargo, folds_purged_embargo = evaluate_cv(dataset_clean, feature_cols_clean, splits_purged_embargo, "purged_kfold_embargo")

fold_results = pd.concat([folds_random, folds_chrono, folds_purged, folds_purged_embargo], ignore_index=True)
predictions = pd.concat([pred_random, pred_chrono, pred_purged, pred_purged_embargo], ignore_index=True)

fold_results

## 15. Cross-validation score comparison

Random K-fold is expected to look better if overlap leakage matters.

In [ ]:
cv_summary = (
    fold_results
    .groupby("method")
    .agg(
        mean_auc=("auc", "mean"),
        std_auc=("auc", "std"),
        mean_accuracy=("accuracy", "mean"),
        mean_log_loss=("log_loss", "mean"),
        mean_train_size=("n_train", "mean"),
        mean_test_size=("n_test", "mean"),
    )
    .reset_index()
    .sort_values("mean_auc", ascending=False)
)

cv_summary

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = cv_summary.sort_values("mean_auc")
plt.barh(plot_df["method"], plot_df["mean_auc"])
plt.axvline(0.5, linestyle="--", label="random classifier")
plt.title("Cross-Validation AUC by Method")
plt.xlabel("Mean fold AUC")
plt.ylabel("Method")
plt.legend()
plt.show()

## 16. Leaky feature demonstration

Now include the intentionally leaky feature.

All CV methods should show suspiciously high performance, because the feature directly uses future returns.

This is not a recommended feature. It is a detector test.

In [ ]:
pred_leaky_random, folds_leaky_random = evaluate_cv(dataset_leaky, feature_cols_leaky, splits_random, "random_kfold_with_leaky_feature")
pred_leaky_purged, folds_leaky_purged = evaluate_cv(dataset_leaky, feature_cols_leaky, splits_purged_embargo, "purged_embargo_with_leaky_feature")

leaky_fold_results = pd.concat([folds_leaky_random, folds_leaky_purged], ignore_index=True)

leaky_cv_summary = (
    leaky_fold_results
    .groupby("method")
    .agg(
        mean_auc=("auc", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_log_loss=("log_loss", "mean"),
    )
    .reset_index()
)

leaky_cv_summary

## 17. Feature availability audit

We flag features with suspicious names or impossible availability.

In production, this should be replaced by a full feature registry with timestamps.

In [ ]:
def feature_availability_audit(feature_cols):
    rows = []
    for col in feature_cols:
        lower = col.lower()
        suspicious = any(token in lower for token in ["future", "forward", "leaky", "target", "label"])
        rows.append({
            "feature": col,
            "flag_name_suggests_future_information": suspicious,
            "availability_assumption": "known_at_prediction_time" if not suspicious else "not_allowed_future_information",
        })
    return pd.DataFrame(rows)

feature_audit_clean = feature_availability_audit(feature_cols_clean)
feature_audit_leaky = feature_availability_audit(feature_cols_leaky)

feature_audit_leaky[feature_audit_leaky["flag_name_suggests_future_information"]]

## 18. Sample uniqueness

When events overlap, each sample is not fully independent.

Average uniqueness for an event is the average of:

$$
u_t = \frac{1}{concurrency_t}
$$

over the event interval.

Lower uniqueness means more duplicated information across labels.

In [ ]:
def sample_uniqueness(event_table, concurrency):
    rows = []

    for idx, row in event_table.iterrows():
        mask = (concurrency.index >= row["t0"]) & (concurrency.index <= row["t1"])
        c = concurrency.loc[mask]
        uniqueness = (1.0 / c.replace(0, np.nan)).mean()
        rows.append({
            "timestamp": idx,
            "t0": row["t0"],
            "t1": row["t1"],
            "avg_uniqueness": uniqueness,
            "event_length_days": int(mask.sum()),
        })

    return pd.DataFrame(rows).set_index("timestamp")

uniqueness = sample_uniqueness(dataset_clean, concurrency)

uniqueness_report = uniqueness["avg_uniqueness"].describe().to_frame("avg_uniqueness")

uniqueness_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(uniqueness["avg_uniqueness"].dropna(), bins=40)
plt.title("Distribution of Average Sample Uniqueness")
plt.xlabel("Average uniqueness")
plt.ylabel("Count")
plt.show()

## 19. Embargo sensitivity

Increasing embargo removes more training data.

We test embargo values and compare AUC and train-size loss.

In [ ]:
embargo_grid = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10]
embargo_rows = []

for embargo in embargo_grid:
    splits = purged_kfold_splits(dataset_clean, config.n_splits, embargo_pct=embargo)
    preds_e, folds_e = evaluate_cv(dataset_clean, feature_cols_clean, splits, f"purged_embargo_{embargo:.3f}")

    diag_e = split_diagnostics(dataset_clean, splits)

    embargo_rows.append({
        "embargo_pct": embargo,
        "mean_auc": folds_e["auc"].mean(),
        "mean_accuracy": folds_e["accuracy"].mean(),
        "mean_log_loss": folds_e["log_loss"].mean(),
        "mean_train_size": folds_e["n_train"].mean(),
        "mean_removed_pct": diag_e["removed_pct"].mean(),
        "remaining_overlaps": diag_e["remaining_train_test_interval_overlaps"].sum(),
    })

embargo_sensitivity = pd.DataFrame(embargo_rows)

embargo_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(embargo_sensitivity["embargo_pct"], embargo_sensitivity["mean_auc"], marker="o")
plt.axhline(0.5, linestyle="--")
plt.title("Embargo Sensitivity: Mean AUC")
plt.xlabel("Embargo percentage")
plt.ylabel("Mean AUC")
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(embargo_sensitivity["embargo_pct"], embargo_sensitivity["mean_train_size"], marker="o")
plt.title("Embargo Sensitivity: Training Set Size")
plt.xlabel("Embargo percentage")
plt.ylabel("Mean train size")
plt.show()

## 20. Strategy backtest from cross-validated predictions

Turn out-of-sample predicted probabilities into positions:

$$
position_t = 2(p_t-0.5)
$$

Then clip to $[-1,1]$.

Returns are the event forward returns, shifted into non-overlapping daily accounting approximately by assigning signal to $t_0$.

This is not a production backtest. It is a CV prediction quality sanity check.

In [ ]:
def prediction_strategy_returns(preds, data, config):
    if preds.empty:
        return pd.DataFrame()

    p = preds.copy()
    p["timestamp"] = pd.to_datetime(p["timestamp"])
    p = p.sort_values("timestamp")

    signal = pd.Series(index=data.index, dtype=float)
    signal.loc[p["timestamp"]] = 2.0 * (p["pred_prob"].to_numpy() - 0.5)
    signal = signal.clip(-1.0, 1.0).fillna(0.0)

    held = signal.shift(1).fillna(0.0)
    gross = held * data["asset_return"]
    turnover = held.diff().abs().fillna(held.abs())
    cost = turnover * config.transaction_cost_bps / 10000.0
    net = gross - cost
    nav = (1 + net).cumprod()

    return pd.DataFrame({
        "signal": signal,
        "held_position": held,
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
    }, index=data.index)

bt_from_predictions = {
    "random_kfold": prediction_strategy_returns(pred_random, data, config),
    "chronological_kfold_no_purge": prediction_strategy_returns(pred_chrono, data, config),
    "purged_kfold_no_embargo": prediction_strategy_returns(pred_purged, data, config),
    "purged_kfold_embargo": prediction_strategy_returns(pred_purged_embargo, data, config),
}

def backtest_metrics(bt, config):
    if bt.empty:
        return {}
    r = bt["net_return"].dropna()
    nav = (1 + r).cumprod()
    _, mdd = max_drawdown(nav)
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)
    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    return {
        "annualised_return": float(ann_return),
        "annualised_vol": float(ann_vol),
        "sharpe_proxy": float(sharpe),
        "max_drawdown": float(mdd),
        "VaR": var,
        "CVaR": cvar,
        "mean_turnover": float(bt["turnover"].mean()),
        "annualised_cost_drag": float(bt["transaction_cost"].mean() * config.annualisation),
        "total_return": float(nav.iloc[-1] - 1),
    }

prediction_backtest_summary = pd.DataFrame([
    {"method": name, **backtest_metrics(bt, config)}
    for name, bt in bt_from_predictions.items()
]).sort_values("sharpe_proxy", ascending=False)

prediction_backtest_summary

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in bt_from_predictions.items():
    if not bt.empty:
        plt.plot(bt.index, bt["nav"], label=name)
plt.title("Prediction-Based Strategy NAV by CV Method")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 21. Fold-level timeline plot

This plot shows where test folds sit in time.

For purged folds, training data around the test interval is removed.

In [ ]:
def fold_timeline_frame(event_table, splits):
    rows = []

    for s in splits:
        fold_id = s["fold_id"]
        test_set = set(s["test_idx"])
        train_set = set(s["train_idx"])

        for i, ts in enumerate(event_table.index):
            if i in test_set:
                status = "test"
            elif i in train_set:
                status = "train"
            else:
                status = "purged_or_embargoed"

            rows.append({
                "fold_id": fold_id,
                "timestamp": ts,
                "position": i,
                "status": status,
            })

    return pd.DataFrame(rows)

timeline_purged_embargo = fold_timeline_frame(dataset_clean, splits_purged_embargo)

timeline_purged_embargo.head()

In [ ]:
status_map = {"train": 0, "purged_or_embargoed": 1, "test": 2}

plt.figure(figsize=(12, 5))
for fold_id, group in timeline_purged_embargo.groupby("fold_id"):
    y = np.full(len(group), fold_id)
    colors = group["status"].map(status_map)
    plt.scatter(group["position"], y, c=colors, s=4)

plt.title("Purged K-Fold with Embargo Timeline")
plt.xlabel("Observation position")
plt.ylabel("Fold")
plt.yticks(sorted(timeline_purged_embargo["fold_id"].unique()))
plt.show()

## 22. Governance flags

The validation process should be reviewed if:

1. random K-fold materially outperforms purged CV;
2. purged train/test overlaps remain;
3. leaky feature dramatically improves performance;
4. embargo removes too much training data;
5. sample uniqueness is low;
6. prediction backtest only works under leaky/random CV.

In [ ]:
random_auc = cv_summary[cv_summary["method"] == "random_kfold"]["mean_auc"].iloc[0]
purged_auc = cv_summary[cv_summary["method"] == "purged_kfold_embargo"]["mean_auc"].iloc[0]
remaining_overlaps = split_diag[split_diag["method"] == "purged_kfold_embargo"]["remaining_train_test_interval_overlaps"].sum()
leaky_auc = leaky_cv_summary["mean_auc"].max()
clean_auc = cv_summary["mean_auc"].max()
mean_uniqueness = uniqueness["avg_uniqueness"].mean()
mean_removed_pct = split_diag[split_diag["method"] == "purged_kfold_embargo"]["removed_pct"].mean()

random_bt_sharpe = prediction_backtest_summary[prediction_backtest_summary["method"] == "random_kfold"]["sharpe_proxy"].iloc[0]
purged_bt_sharpe = prediction_backtest_summary[prediction_backtest_summary["method"] == "purged_kfold_embargo"]["sharpe_proxy"].iloc[0]

governance_flags = pd.DataFrame([{
    "random_kfold_auc": random_auc,
    "purged_embargo_auc": purged_auc,
    "random_minus_purged_auc": random_auc - purged_auc,
    "remaining_purged_overlaps": int(remaining_overlaps),
    "leaky_feature_best_auc": leaky_auc,
    "clean_feature_best_auc": clean_auc,
    "mean_sample_uniqueness": mean_uniqueness,
    "mean_removed_pct_purged_embargo": mean_removed_pct,
    "random_prediction_backtest_sharpe": random_bt_sharpe,
    "purged_prediction_backtest_sharpe": purged_bt_sharpe,
    "flag_random_cv_much_better_than_purged": bool(random_auc - purged_auc > 0.05),
    "flag_remaining_overlaps_after_purge": bool(remaining_overlaps > 0),
    "flag_leaky_feature_suspiciously_good": bool(leaky_auc - clean_auc > 0.10),
    "flag_low_sample_uniqueness": bool(mean_uniqueness < 0.50),
    "flag_embargo_removes_more_than_25pct": bool(mean_removed_pct > 0.25),
    "flag_random_backtest_much_better": bool(random_bt_sharpe - purged_bt_sharpe > 0.75),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_random_cv_much_better_than_purged",
        "flag_remaining_overlaps_after_purge",
        "flag_leaky_feature_suspiciously_good",
        "flag_low_sample_uniqueness",
        "flag_embargo_removes_more_than_25pct",
        "flag_random_backtest_much_better",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. every event has $t_1>t_0$;
2. purged folds have zero train/test interval overlaps;
3. embargoed folds have non-empty training sets;
4. feature matrix contains no NaNs after dataset construction;
5. clean feature set excludes leaky features;
6. leaky audit flags leaky feature;
7. prediction probabilities are in $[0,1]$;
8. fold predictions cover each test observation once per method.

In [ ]:
audit_rows = []

valid_event_times = bool((dataset_clean["t1"] > dataset_clean["t0"]).all())
audit_rows.append({
    "check": "event_t1_after_t0",
    "value": valid_event_times,
    "passed": valid_event_times,
})

purged_zero_overlaps = bool(
    split_diag[split_diag["method"] == "purged_kfold_embargo"]["remaining_train_test_interval_overlaps"].sum() == 0
)
audit_rows.append({
    "check": "purged_embargo_has_zero_train_test_overlaps",
    "value": purged_zero_overlaps,
    "passed": purged_zero_overlaps,
})

nonempty_train = bool((split_diag[split_diag["method"] == "purged_kfold_embargo"]["n_train"] > config.min_train_size).all())
audit_rows.append({
    "check": "purged_embargo_train_sets_large_enough",
    "value": nonempty_train,
    "passed": nonempty_train,
})

features_no_nan = bool(dataset_clean[feature_cols_clean].notna().all().all())
audit_rows.append({
    "check": "clean_features_no_nan",
    "value": features_no_nan,
    "passed": features_no_nan,
})

clean_excludes_leaky = bool(not any(c.startswith("LEAKY") for c in feature_cols_clean))
audit_rows.append({
    "check": "clean_feature_set_excludes_leaky_features",
    "value": clean_excludes_leaky,
    "passed": clean_excludes_leaky,
})

leaky_flagged = bool(feature_audit_leaky["flag_name_suggests_future_information"].any())
audit_rows.append({
    "check": "feature_audit_flags_leaky_feature",
    "value": leaky_flagged,
    "passed": leaky_flagged,
})

pred_probs_valid = bool(((predictions["pred_prob"] >= 0) & (predictions["pred_prob"] <= 1)).all())
audit_rows.append({
    "check": "prediction_probabilities_in_unit_interval",
    "value": pred_probs_valid,
    "passed": pred_probs_valid,
})

coverage_rows = []
for method, group in predictions.groupby("method"):
    duplicate_count = group.duplicated(["timestamp"]).sum()
    coverage_rows.append({"method": method, "duplicates": int(duplicate_count)})
coverage_table = pd.DataFrame(coverage_rows)
no_duplicates = bool((coverage_table["duplicates"] == 0).all())
audit_rows.append({
    "check": "one_prediction_per_timestamp_per_method",
    "value": no_duplicates,
    "passed": no_duplicates,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 24. Practical checklist for purged CV

Before trusting cross-validation for a financial ML model:

1. **Define event intervals**  
   Every label needs $t_0$ and $t_1$.

2. **Check label overlap**  
   Forward-return labels usually overlap.

3. **Avoid random K-fold**  
   It mixes overlapping events across train and test.

4. **Purge overlaps**  
   Remove training events whose intervals intersect test events.

5. **Apply embargo**  
   Remove observations after test folds to reduce boundary leakage.

6. **Fit preprocessing inside folds**  
   Scaling and feature selection must be train-fold-only.

7. **Audit feature availability**  
   No future-derived features.

8. **Track sample uniqueness**  
   Highly overlapping events reduce independent evidence.

9. **Run sensitivity to embargo**  
   Results should not depend on one arbitrary embargo setting.

10. **Compare CV methods**  
   Random K-fold being much better is a warning, not a win.

## 25. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/purged_kfold_and_embargo_cv_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "data": output_dir / "synthetic_time_series_data.csv",
    "features_all": output_dir / "features_all.csv",
    "dataset_clean": output_dir / "event_dataset_clean.csv",
    "dataset_leaky": output_dir / "event_dataset_leaky.csv",
    "feature_audit_clean": output_dir / "feature_audit_clean.csv",
    "feature_audit_leaky": output_dir / "feature_audit_leaky.csv",
    "concurrency": output_dir / "concurrency.csv",
    "concurrency_report": output_dir / "concurrency_report.csv",
    "split_diagnostics": output_dir / "split_diagnostics.csv",
    "fold_results": output_dir / "fold_results.csv",
    "cv_summary": output_dir / "cv_summary.csv",
    "predictions": output_dir / "cv_predictions.csv",
    "leaky_fold_results": output_dir / "leaky_fold_results.csv",
    "leaky_cv_summary": output_dir / "leaky_cv_summary.csv",
    "uniqueness": output_dir / "sample_uniqueness.csv",
    "uniqueness_report": output_dir / "uniqueness_report.csv",
    "embargo_sensitivity": output_dir / "embargo_sensitivity.csv",
    "prediction_backtest_summary": output_dir / "prediction_backtest_summary.csv",
    "timeline_purged_embargo": output_dir / "timeline_purged_embargo.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

data.to_csv(paths["data"])
features_all.to_csv(paths["features_all"])
dataset_clean.to_csv(paths["dataset_clean"])
dataset_leaky.to_csv(paths["dataset_leaky"])
feature_audit_clean.to_csv(paths["feature_audit_clean"], index=False)
feature_audit_leaky.to_csv(paths["feature_audit_leaky"], index=False)
concurrency.to_frame("concurrency").to_csv(paths["concurrency"])
concurrency_report.to_frame("value").to_csv(paths["concurrency_report"])
split_diag.to_csv(paths["split_diagnostics"], index=False)
fold_results.to_csv(paths["fold_results"], index=False)
cv_summary.to_csv(paths["cv_summary"], index=False)
predictions.to_csv(paths["predictions"], index=False)
leaky_fold_results.to_csv(paths["leaky_fold_results"], index=False)
leaky_cv_summary.to_csv(paths["leaky_cv_summary"], index=False)
uniqueness.to_csv(paths["uniqueness"])
uniqueness_report.to_csv(paths["uniqueness_report"])
embargo_sensitivity.to_csv(paths["embargo_sensitivity"], index=False)
prediction_backtest_summary.to_csv(paths["prediction_backtest_summary"], index=False)
timeline_purged_embargo.to_csv(paths["timeline_purged_embargo"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "purged_kfold_and_embargo_cv_outputs",
    "schema_version": "purged_kfold_and_embargo_cv_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "sklearn_available": SKLEARN_AVAILABLE,
    "n_clean_events": int(len(dataset_clean)),
    "n_clean_features": int(len(feature_cols_clean)),
    "n_leaky_features": int(len(feature_cols_leaky)),
    "cv_summary": cv_summary.to_dict(orient="records"),
    "leaky_cv_summary": leaky_cv_summary.to_dict(orient="records"),
    "concurrency_report": concurrency_report.to_dict(),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "Labels are H-day overlapping forward-return events with t0 and t1 timestamps.",
        "Purging removes training samples whose event intervals overlap validation intervals.",
        "Embargo removes samples after validation folds.",
        "Random K-fold is included as a failure-mode comparison.",
        "An intentionally leaky feature is included to test feature-availability diagnostics.",
        "Feature scaling and model fitting happen inside each fold.",
        "This notebook is about validation hygiene, not alpha discovery."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["cv_summary"], paths["prediction_backtest_summary"], paths["governance_flags"], paths["audit_report"], paths["manifest"]

## 26. Limitations

### Synthetic data

The dataset is synthetic. Real data requires corporate-action handling, delisting controls, futures rolls, exchange calendars, and timestamped feature availability.

### Simple model

The classifier is intentionally simple. The focus is validation hygiene, not machine-learning sophistication.

### Fixed horizon labels

Only one fixed-horizon label is used. Real event labels may have variable end times, profit-taking barriers, stop-loss barriers, or vertical barriers.

### Basic embargo

The embargo is percentage-based. Real embargo length should reflect feature lookbacks, label horizon, market microstructure, and information latency.

### No combinatorial purged CV

This notebook implements standard purged K-fold. Combinatorial Purged Cross-Validation is a more advanced extension.

### Approximate prediction backtest

The prediction-based strategy is a sanity check, not a production backtest.

## 27. Research frontier and extensions

Important extensions include:

1. triple-barrier event labelling;
2. sample uniqueness weighting;
3. sequential bootstrap;
4. combinatorial purged cross-validation;
5. purged cross-validation for meta-labelling;
6. feature registry with availability timestamps;
7. fold-aware feature selection;
8. nested purged model selection;
9. purged CV for panel data across assets;
10. Chinese futures labels with night sessions, contract rolls, and exchange-specific trading breaks.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_08_parameter_sensitivity_and_overfitting**  
   Study parameter surfaces and overfit diagnostics.

2. **05_09_triple_barrier_labeling_and_meta_labeling**  
   Build event labels with variable horizons.

3. **05_10_research_audit_trail_and_manifest**  
   Store feature availability and CV splits.

4. **05_11_live_shadow_mode_monitoring**  
   Compare CV expectations to paper/live outcomes.

5. **06_05_microstructure_feature_validation**  
   Validate L1/L2 features with strict timing.

6. **07_07_ml_alpha_validation_capstone**  
   Full ML alpha validation with purging, embargo, and audit trails.

## 29. Summary

This notebook implemented purged K-fold and embargo cross-validation.

It showed how to:

1. simulate time-series data;
2. engineer features with availability discipline;
3. construct overlapping forward-return labels;
4. store event intervals $t_0,t_1$;
5. measure label concurrency;
6. compute sample uniqueness;
7. implement random K-fold;
8. implement chronological K-fold;
9. implement purged K-fold;
10. implement purged K-fold with embargo;
11. count train/test interval overlaps;
12. train models fold-by-fold without preprocessing leakage;
13. compare CV methods;
14. demonstrate leaky-feature failure;
15. audit feature availability;
16. test embargo sensitivity;
17. create prediction-based backtests;
18. create governance flags and audit checks;
19. save outputs and manifests.

The key computational takeaway:

> Purging requires event intervals, not just row numbers.

The key financial takeaway:

> Random K-fold can make a trading model look good simply because train and test labels share the same future returns.

## 30. Further reading

- López de Prado, *Advances in Financial Machine Learning.*
- Literature on purged K-fold cross-validation and embargo.
- Research on triple-barrier labelling and meta-labelling.
- Bailey et al. on backtest overfitting.
- Institutional model-validation literature on feature availability, timestamping, and data leakage.